# 09 - Safe USB opcode probing

The SCiO apps expose no command that reads firmware or memory out of the device.
Three opcodes are *declared* by the firmware but never used by any app, and the
file-list handler reserves a band of opcodes (87-95) that are mostly unclaimed.
This notebook probes exactly that surface, read-only and safety-gated, to check
whether any of it returns useful data.

**Safety.** `scio.probe.ScioProbe` sends only allowlisted read/query opcodes with
empty payloads (plus a file id / small parameter ids). It refuses every write or
state-changing opcode. The reserved band is opt-in and sent one opcode at a time.

**Result on this device (fw 147):** confirmed negative - `0x06/0x08/0x09` do not
respond, extended `READ_FILE_HEADER` is ignored, `0x88/0x89/0x93/0x95` ack empty,
`0x8A-0x8F` are unimplemented. Re-run if you have a different firmware.

Turn the SCiO on to **steady blue** (power-cycle if it only pulses) before running.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
from scio import usb, probe, protocol
from scio.decode import entropy

## 1. Connect

In [ ]:
port = next((p["device"] for p in usb.find_scio_ports() if p["is_scio"]), None)
assert port, "SCiO not found (VID:PID 0451:16AA). Power-cycle to steady blue and replug."
dev = usb.ScioUSB(port, timeout=2.0).open()
info = dev.read_device_info()
print("device", info["device_id"], "fw", info["firmware_version"])

## 2. Safe sweep (never touches the reserved band)

In [ ]:
pr = probe.ScioProbe(dev)
pr.run_safe_sweep(include_parameter_get=True, include_file_header_ext=True)

print("Probes that returned data:")
rows = probe.summarize(pr.results)
if not rows:
    print("  (none)")
for r in rows:
    print(f"  {r['label']:46s} len={r['len']:>4} {r['hex_head']}")

## 3. Interpret the file headers

The extended `READ_FILE_HEADER` probes also give the exact size of each file
(`u32 type, size, version, checksum`). `dsp_op` size tells a future flash/JTAG
dump how much to expect.

In [ ]:
import struct
for r in pr.results:
    if r.command == 0x87 and r.response_len == 16:
        t, size, ver, cs = struct.unpack("<4I", bytes.fromhex(r.response_hex)[:16])
        name = protocol.FIRMWARE_FILE_NAMES.get(t, str(t))
        print(f"  {name:18s} size={size:6d}  version={ver:3d}  checksum={cs}")
        # one line per file id is enough; break duplicates


## 4. Reserved-band sweep (opt-in)

Undocumented opcodes in the 87-95 band. **Small risk**: an unknown opcode could
be a destructive/bootloader command. This cell checks the device is still alive
after each opcode and stops if not. Set `RUN_RESERVED = True` to enable.

In [ ]:
RUN_RESERVED = False

if RUN_RESERVED:
    from scio.protocol import Cmd
    for cmd in probe.RESERVED_BAND:
        r = pr.probe(cmd, allow_reserved=True)
        print(f"  0x{cmd:02X}: ok={r.ok} len={r.response_len} note={r.note} {r.response_hex[:40]}")
        try:
            if dev._command(Cmd.READ_DEVICE_STATUS).command != 0x00:
                print("  !! unexpected reply; stopping"); break
        except Exception as e:
            print(f"  !! device stopped responding after 0x{cmd:02X}: {e}"); break
    print("final sanity:", {k: round(v, 1) for k, v in dev.read_temperature().items()})
else:
    print("Reserved-band sweep disabled. Set RUN_RESERVED = True to run it.")

## 5. Analyse any bulk data (only if something returned a file body)

In [ ]:
from scio import firmware, keyrecover, store
big = [r for r in pr.results if r.ok and (r.response_len or 0) > 64]
if big:
    for r in big:
        data = bytes.fromhex(r.response_hex)
        print(r.label, "len", r.response_len, "entropy", round(entropy(data), 2))
    # If one is a firmware body, save it and run key recovery:
    #   Path("01_rawdata/device_files/dsp_op.bin").write_bytes(<body>)
    #   blobs = firmware.load_blob_dir("01_rawdata/device_files")
    #   scans = [store.load_scan(p) for p in Path("01_rawdata/scan_json").glob("scan_*.json")]
    #   print(keyrecover.recover(scans, firmware_blobs=blobs, device=info).conclusion)
else:
    print("No bulk data returned - USB-only firmware/key extraction is not possible.")
    print("Next: hardware routes (external SPI flash dump, or BF512 JTAG). See documentation/firmware_notes.md")

In [ ]:
dev.close()
print("closed")